# 🛸 Byzantine Attack Detection in Drone Swarms
### Spatio-Temporal GAT+GRU with Explainability — Google Colab Training Notebook

---

**Pipeline stages:**
1. 🔧 Environment Setup (GPU + dependencies)
2. 📁 Upload / Mount project files
3. 📊 Data Download & Preprocessing
4. 🕸️ Graph Construction & Attack Injection
5. 🏋️ Train Baselines (MLP / LSTM / CNN / GCN)
6. 🏋️ Train Proposed Model (GAT+GRU)
7. 📈 Evaluation & Results
8. 🔍 Explainability
9. 🧪 Ablation Study
10. 💾 Download Results

> **Tip:** Runtime → Change runtime type → **GPU (T4)** before running.

## 🔧 Stage 0 — Environment Setup

In [ ]:
# ── Check GPU ──────────────────────────────────────────────────────────────
import subprocess, sys

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print('⚠️  No GPU detected. Go to Runtime → Change runtime type → GPU.')

import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Install PyTorch Geometric (matches Colab's default torch+CUDA) ─────────
import torch
torch_ver = torch.__version__.split('+')[0]   # e.g. '2.3.0'
cuda_tag  = 'cu121' if torch.cuda.is_available() else 'cpu'

print(f'Installing PyG for torch=={torch_ver}, {cuda_tag} ...')

!pip install -q torch-geometric
!pip install -q torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-{torch_ver}+{cuda_tag}.html

# Other dependencies
!pip install -q pyarrow shap networkx seaborn tqdm

print('\n✅ All dependencies installed.')

## 📁 Stage 1 — Upload Project Files

Choose **one** method below (A = Google Drive, B = ZIP upload).

In [ ]:
# ── METHOD A: Mount Google Drive ───────────────────────────────────────────
# Upload your project folder to Google Drive first, then run this cell.
# Adjust DRIVE_PROJECT_PATH to where you placed the project folder.

from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT_PATH = '/content/drive/MyDrive/iod'   # ← change if needed

import os, shutil
if os.path.exists(DRIVE_PROJECT_PATH):
    # Symlink so imports work from /content/iod
    if not os.path.exists('/content/iod'):
        os.symlink(DRIVE_PROJECT_PATH, '/content/iod')
    print(f'✅ Project linked from Drive: {DRIVE_PROJECT_PATH}')
else:
    print(f'❌ Path not found: {DRIVE_PROJECT_PATH}')
    print('   Upload the project folder to Google Drive and update DRIVE_PROJECT_PATH.')

In [ ]:
# ── METHOD B: Upload ZIP ───────────────────────────────────────────────────
# Zip your project folder on Windows:
#   Right-click the 'iod' folder → Send to → Compressed (zipped) folder
# Then run this cell and choose the zip file.

from google.colab import files
import zipfile, os

print('Select your iod.zip file...')
uploaded = files.upload()          # opens file picker

for fname in uploaded:
    print(f'Extracting {fname} ...')
    with zipfile.ZipFile(fname, 'r') as zf:
        zf.extractall('/content/')

# Rename extracted folder to /content/iod if needed
extracted = [d for d in os.listdir('/content') if os.path.isdir(f'/content/{d}')
             and d not in ('sample_data', 'drive')]
print('Extracted folders:', extracted)
print('✅ Upload complete.')

In [ ]:
# ── Set project root & verify structure ───────────────────────────────────
import os, sys
from pathlib import Path

# Update this if your folder extracted with a different name
PROJECT_ROOT = Path('/content/iod')

if not PROJECT_ROOT.exists():
    # Try auto-detect
    candidates = [p for p in Path('/content').iterdir()
                  if p.is_dir() and (p/'src').exists()]
    if candidates:
        PROJECT_ROOT = candidates[0]
        print(f'Auto-detected project root: {PROJECT_ROOT}')
    else:
        raise FileNotFoundError(
            'Project folder not found at /content/iod. '
            'Run Method A or B above first.')

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

# Verify key directories
required = ['src', 'models', 'attacks', 'graphs', 'evaluation', 'scripts']
for d in required:
    status = '✅' if (PROJECT_ROOT / d).exists() else '❌'
    print(f'  {status}  {d}/')

print(f'\nWorking directory: {os.getcwd()}')

In [ ]:
# ── Environment variables for the session ─────────────────────────────────
import os
os.environ['PYTHONIOENCODING'] = 'utf-8'

# Confirm GPU is visible to PyTorch
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training device: {device}')
if device.type == 'cuda':
    print(f'GPU memory     : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## 📊 Stage 2 — Data Download & Preprocessing

In [ ]:
# ── Download the UAV dataset from GitHub ──────────────────────────────────
from pathlib import Path
import subprocess

raw_dir = Path('data/raw')
raw_dir.mkdir(parents=True, exist_ok=True)

dataset_dir = raw_dir / 'UAVs-Dataset-Under-Normal-and-Cyberattacks'
if not dataset_dir.exists():
    print('Cloning UAV dataset (~60 MB)...')
    result = subprocess.run(
        ['git', 'clone', '--depth=1',
         'https://github.com/uamughal/UAVs-Dataset-Under-Normal-and-Cyberattacks.git',
         str(dataset_dir)],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(result.stderr)
    else:
        print('✅ Dataset downloaded.')
else:
    print('✅ Dataset already present.')

# Verify CSV
csv_path = dataset_dir / 'Dataset_T-ITS.csv'
if csv_path.exists():
    import pandas as pd
    df_check = pd.read_csv(csv_path, nrows=5)
    print(f'CSV columns: {list(df_check.columns[:5])} ...')
    print(f'CSV found at: {csv_path}')
else:
    print(f'❌ CSV not found at {csv_path}')

In [ ]:
# ── Run preprocessing ──────────────────────────────────────────────────────
!python scripts/preprocess.py

from pathlib import Path
for split in ['train', 'val', 'test']:
    p = Path(f'data/processed/{split}.parquet')
    status = '✅' if p.exists() else '❌'
    size = f'{p.stat().st_size/1024:.0f} KB' if p.exists() else 'MISSING'
    print(f'  {status}  {split}.parquet  ({size})')

In [ ]:
# ── Quick EDA ──────────────────────────────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

train_df = pd.read_parquet('data/processed/train.parquet')
print(f'Train shape: {train_df.shape}')
print(f'Columns    : {list(train_df.columns[:8])} ...')
print('\nClass distribution:')
if 'attack_type' in train_df.columns:
    print(train_df['attack_type'].value_counts())
elif 'label' in train_df.columns:
    print(train_df['label'].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.patch.set_facecolor('#1a1a2e')
for ax in axes:
    ax.set_facecolor('#16213e')
    ax.tick_params(colors='white'); ax.xaxis.label.set_color('white')
    ax.yaxis.label.set_color('white'); ax.title.set_color('white')

# Class balance
col = 'attack_type' if 'attack_type' in train_df.columns else 'label'
train_df[col].value_counts().plot(kind='bar', ax=axes[0], color='#6C63FF', edgecolor='white')
axes[0].set_title('Class Distribution (Train)')
axes[0].tick_params(axis='x', rotation=30)

# Feature missing values
feat_cols = [c for c in train_df.columns if c not in [col, 'is_attack', 'domain']]
missing = train_df[feat_cols[:20]].isnull().mean() * 100
missing.plot(kind='bar', ax=axes[1], color='#00D4FF', edgecolor='white')
axes[1].set_title('Missing Values % (first 20 features)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('outputs/figures/colab_eda.png', dpi=120, bbox_inches='tight',
            facecolor='#1a1a2e')
plt.show()
print('✅ EDA plots saved.')

## 🕸️ Stage 3 — Graph Construction & Attack Injection

> **Tip:** Adjust `N_SNAPSHOTS` below. Use `200` for a quick run, `1000` for full training.

In [ ]:
# ── Training configuration ─────────────────────────────────────────────────
# Adjust these before running the pipeline

N_SNAPSHOTS  = 200    # swarm snapshots: 200 = fast smoke test, 1000 = full
N_EPOCHS     = 50     # training epochs: 50 = quick, 100-200 = full
N_EVAL_SEEDS = 5      # evaluation seeds: 1 = single run, 5 = mean±std
ATTACK_TYPE  = 'false_state'   # false_state | intermittent | colluding | delay
GRAPH_TYPE   = 'knn'           # knn | distance | hexagonal
BATCH_SIZE   = 64              # increase on GPU for speed

print(f'Config:')
print(f'  Snapshots  : {N_SNAPSHOTS}')
print(f'  Epochs     : {N_EPOCHS}')
print(f'  Eval seeds : {N_EVAL_SEEDS}')
print(f'  Attack type: {ATTACK_TYPE}')
print(f'  Graph type : {GRAPH_TYPE}')
print(f'  Batch size : {BATCH_SIZE}')

In [ ]:
# ── Build swarm graphs with attack injection ───────────────────────────────
!python scripts/build_graphs.py \
    --attack_type {ATTACK_TYPE} \
    --graph_type  {GRAPH_TYPE} \
    --n_snapshots {N_SNAPSHOTS}

from pathlib import Path
graph_files = list(Path('data/processed').glob('graphs_*.pkl'))
print(f'\nGraph files generated: {len(graph_files)}')
for gf in graph_files:
    print(f'  ✅  {gf.name}  ({gf.stat().st_size/1024:.0f} KB)')

## 🏋️ Stage 5 — Train Baseline Models

In [ ]:
# ── Train MLP, LSTM, 1D-CNN, GCN baselines ────────────────────────────────
!python scripts/train_baselines.py \
    --attack_type {ATTACK_TYPE} \
    --epochs      {N_EPOCHS} \
    --snapshots   {N_SNAPSHOTS}

In [ ]:
# ── Verify baseline checkpoints ───────────────────────────────────────────
import json
from pathlib import Path

model_dir = Path('outputs/models')
print(f"{'Model':<8}  {'Ckpt':>4}  {'Val Loss':>9}  {'Best Val F1':>11}  {'Threshold':>9}")
print('-' * 52)
for name in ['mlp', 'lstm', 'cnn', 'gcn']:
    ckpt   = model_dir / f'best_{name}.pt'
    hist_p = model_dir / f'history_{name}.json'
    thr_p  = model_dir / f'threshold_{name}.json'

    status = '✅' if ckpt.exists() else '❌'

    val_loss = '—'
    best_f1  = '—'
    if hist_p.exists():
        h = json.loads(hist_p.read_text())
        last = h.get('val_loss', [None])[-1]
        if last is not None:
            val_loss = f'{last:.4f}'
        f1_hist = h.get('val_f1', [])
        if f1_hist:
            best_f1 = f'{max(f1_hist):.4f}'

    thresh = '—'
    if thr_p.exists():
        t = json.loads(thr_p.read_text())
        thresh = f'{t.get("threshold", 0.5):.3f}'

    print(f'  {status}  {name:<8}  {val_loss:>9}  {best_f1:>11}  {thresh:>9}')

In [ ]:
# ── Plot baseline training curves ─────────────────────────────────────────
import json, matplotlib.pyplot as plt
from pathlib import Path

model_dir = Path('outputs/models')
palette = {'mlp': '#FFB300', 'lstm': '#FF7043', 'cnn': '#00E676', 'gcn': '#F06292'}

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
fig.patch.set_facecolor('#1a1a2e')
fig.suptitle('Baseline Training Curves', color='white', fontsize=14)

for ax in axes:
    ax.set_facecolor('#16213e')
    ax.tick_params(colors='white')
    ax.xaxis.label.set_color('white')
    ax.yaxis.label.set_color('white')
    for spine in ax.spines.values():
        spine.set_edgecolor('#2D3250')

for name, color in palette.items():
    hist_p = model_dir / f'history_{name}.json'
    if not hist_p.exists():
        continue
    h = json.loads(hist_p.read_text())
    epochs_loss = range(1, len(h.get('train_loss', [])) + 1)
    epochs_f1   = range(1, len(h.get('val_f1',    [])) + 1)
    if h.get('train_loss'):
        axes[0].plot(epochs_loss, h['train_loss'], label=name, color=color, linewidth=2)
    if h.get('val_loss'):
        axes[1].plot(epochs_loss, h['val_loss'],   label=name, color=color, linewidth=2, linestyle='--')
    if h.get('val_f1'):
        axes[2].plot(epochs_f1,   h['val_f1'],     label=name, color=color, linewidth=2)

axes[0].set_title('Train Loss',  color='white'); axes[0].set_xlabel('Epoch')
axes[1].set_title('Val Loss',    color='white'); axes[1].set_xlabel('Epoch')
axes[2].set_title('Val F1',      color='white'); axes[2].set_xlabel('Epoch')
axes[2].set_ylim(0, 1)

for ax in axes:
    ax.legend(facecolor='#2D3250', labelcolor='white', framealpha=0.8)

plt.tight_layout()
plt.savefig('outputs/figures/colab_baseline_curves.png', dpi=120,
            bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print('✅ Training curves saved.')

## 🏋️ Stage 6 — Train Proposed Temporal GNN Models

In [ ]:
# ── Train GAT+GRU (proposed model) ────────────────────────────────────────
!python scripts/train_gnn_temporal.py \
    --model       gat \
    --attack_type {ATTACK_TYPE} \
    --graph_type  {GRAPH_TYPE} \
    --epochs      {N_EPOCHS} \
    --snapshots   {N_SNAPSHOTS}

In [ ]:
# ── (Optional) Train GraphSAGE+GRU variant ────────────────────────────────
!python scripts/train_gnn_temporal.py \
    --model       graphsage \
    --attack_type {ATTACK_TYPE} \
    --graph_type  {GRAPH_TYPE} \
    --epochs      {N_EPOCHS} \
    --snapshots   {N_SNAPSHOTS}

In [ ]:
# ── Plot GAT+GRU vs baselines training curves ──────────────────────────────
import json, matplotlib.pyplot as plt
from pathlib import Path

model_dir = Path('outputs/models')
palette = {
    'gat_temporal': '#6C63FF',
    'graphsage_temporal': '#00D4FF',
    'mlp':  '#FFB300',
    'lstm': '#FF7043',
    'cnn':  '#00E676',
    'gcn':  '#F06292',
}

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.patch.set_facecolor('#1a1a2e')
fig.suptitle('All Models — Training Curves', color='white', fontsize=14, fontweight='bold')

for ax in axes:
    ax.set_facecolor('#16213e')
    ax.tick_params(colors='white')
    ax.xaxis.label.set_color('white'); ax.yaxis.label.set_color('white')
    for sp in ax.spines.values(): sp.set_edgecolor('#2D3250')

for name, color in palette.items():
    hist_p = model_dir / f'history_{name}.json'
    if not hist_p.exists(): continue
    h = json.loads(hist_p.read_text())
    lw = 3 if 'gat' in name or 'graphsage' in name else 1.5
    ls = '-' if 'gat' in name else ('--' if 'graphsage' in name else ':')
    epochs = range(1, len(h.get('val_loss', [])) + 1)
    if h.get('val_loss'):
        axes[0].plot(epochs, h['val_loss'], label=name, color=color,
                     linewidth=lw, linestyle=ls)
    if h.get('val_f1'):
        axes[1].plot(range(1, len(h['val_f1'])+1), h['val_f1'], label=name,
                     color=color, linewidth=lw, linestyle=ls)

axes[0].set_title('Validation Loss', color='white'); axes[0].set_xlabel('Epoch')
axes[1].set_title('Validation F1',   color='white'); axes[1].set_xlabel('Epoch')
axes[1].set_ylim(0, 1)

for ax in axes:
    ax.legend(facecolor='#2D3250', labelcolor='white', framealpha=0.8, fontsize=8)

plt.tight_layout()
plt.savefig('outputs/figures/colab_all_training_curves.png', dpi=150,
            bbox_inches='tight', facecolor='#1a1a2e')
plt.show()

## 📈 Stage 7 — Evaluation

In [ ]:
# ── Run full evaluation (with val-threshold calibration) ───────────────────
!python scripts/evaluate.py \
    --attack_type {ATTACK_TYPE} \
    --graph_type  {GRAPH_TYPE} \
    --snapshots   {N_SNAPSHOTS} \
    --eval_seeds  {N_EVAL_SEEDS}

In [ ]:
# ── Display results table ─────────────────────────────────────────────────
import json, pandas as pd
from pathlib import Path
from IPython.display import display

tables_dir = Path('outputs/tables')
metrics_file = (
    tables_dir / f'{ATTACK_TYPE}_metrics_seedavg.json'
    if N_EVAL_SEEDS > 1
    else tables_dir / f'{ATTACK_TYPE}_metrics.json'
)

if metrics_file.exists():
    all_metrics = json.loads(metrics_file.read_text())

    rows = []
    for model, md in all_metrics.items():
        if not isinstance(md, dict):
            continue

        if N_EVAL_SEEDS > 1:
            rows.append({
                'Model'      : model,
                'Threshold'  : round(md.get('threshold', 0.5), 3),
                'Accuracy'   : f"{md.get('accuracy', 0):.4f}±{md.get('accuracy_std', 0):.4f}",
                'Precision'  : f"{md.get('precision', 0):.4f}±{md.get('precision_std', 0):.4f}",
                'Recall'     : f"{md.get('recall', 0):.4f}±{md.get('recall_std', 0):.4f}",
                'F1'         : f"{md.get('f1', 0):.4f}±{md.get('f1_std', 0):.4f}",
                'ROC-AUC'    : f"{md.get('roc_auc', 0):.4f}±{md.get('roc_auc_std', 0):.4f}",
                'FPR'        : f"{md.get('fpr', 0):.4f}±{md.get('fpr_std', 0):.4f}",
                'MCC'        : f"{md.get('mcc', 0):.4f}±{md.get('mcc_std', 0):.4f}",
                'BalancedAcc': f"{md.get('balanced_accuracy', 0):.4f}±{md.get('balanced_accuracy_std', 0):.4f}",
            })
        else:
            rows.append({
                'Model'      : model,
                'Threshold'  : round(md.get('threshold', 0.5), 3),
                'Accuracy'   : round(md.get('accuracy', 0), 4),
                'Precision'  : round(md.get('precision', 0), 4),
                'Recall'     : round(md.get('recall', 0), 4),
                'F1'         : round(md.get('f1', 0), 4),
                'ROC-AUC'    : round(md.get('roc_auc', 0), 4),
                'FPR'        : round(md.get('fpr', 0), 4),
                'MCC'        : round(md.get('mcc', 0), 4),
                'BalancedAcc': round(md.get('balanced_accuracy', 0), 4),
            })

    if rows:
        df_res = pd.DataFrame(rows).reset_index(drop=True)
        display(df_res.style
                .set_caption(f'Results — {ATTACK_TYPE.replace("_"," ").title()} Attack'))
    else:
        print('No model results in metrics file.')
else:
    print(f'Metrics file not found: {metrics_file}')
    print('Run evaluation script first.')

In [ ]:
# ── Bar chart comparison ───────────────────────────────────────────────────
import json, matplotlib.pyplot as plt, numpy as np
from pathlib import Path

metrics_file = (
    Path('outputs/tables') / f'{ATTACK_TYPE}_metrics_seedavg.json'
    if N_EVAL_SEEDS > 1
    else Path('outputs/tables') / f'{ATTACK_TYPE}_metrics.json'
)
if not metrics_file.exists():
    print('Run evaluate.py first.'); raise SystemExit

all_metrics = json.loads(metrics_file.read_text())
model_names = [m for m, v in all_metrics.items() if isinstance(v, dict)]
metric_keys  = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
metric_labels= ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']

PALETTE = {
    'GAT+GRU': '#6C63FF', 'GraphSAGE+GRU': '#00D4FF',
    'MLP': '#FFB300',  'LSTM': '#FF7043',
    '1D-CNN': '#00E676', 'GCN': '#F06292',
}
colors = [PALETTE.get(m, '#888') for m in model_names]

x  = np.arange(len(model_names))
w  = 0.15

fig, ax = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor('#1a1a2e'); ax.set_facecolor('#16213e')
ax.tick_params(colors='white'); ax.xaxis.label.set_color('white')
ax.yaxis.label.set_color('white'); ax.title.set_color('white')
for sp in ax.spines.values(): sp.set_edgecolor('#2D3250')

metric_colors = ['#6C63FF','#00D4FF','#FFB300','#FF7043','#00E676']
for i, (mk, ml, mc) in enumerate(zip(metric_keys, metric_labels, metric_colors)):
    vals = [all_metrics.get(m, {}).get(mk, 0) for m in model_names]
    ax.bar(x + i * w, vals, w, label=ml, color=mc, alpha=0.85, edgecolor='white', linewidth=0.3)

ax.set_xticks(x + w * 2)
ax.set_xticklabels(model_names, rotation=20, ha='right', color='white')
ax.set_ylim(0, 1.12); ax.set_ylabel('Score', color='white')
ax.set_title(f'Model Comparison — {ATTACK_TYPE.replace("_"," ").title()} Attack', fontsize=13)
ax.axhline(y=0.9, color='#555', linestyle='--', linewidth=0.8)
ax.legend(facecolor='#2D3250', labelcolor='white', framealpha=0.8)

plt.tight_layout()
plt.savefig('outputs/figures/colab_model_comparison.png', dpi=150,
            bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print('✅ Comparison chart saved.')

In [ ]:
# ── Confusion matrices for all models ─────────────────────────────────────
# Confusion matrix PNGs are saved by evaluate.py; we load and tile them here.
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
from pathlib import Path

fig_dir  = Path('outputs/figures')
cm_files = sorted(fig_dir.glob(f'{ATTACK_TYPE}_cm_*.png'))

if not cm_files:
    print('No confusion matrix PNGs found. Run evaluate.py first.')
else:
    ncols = min(3, len(cm_files))
    nrows = (len(cm_files) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4.5, nrows * 4.5))
    fig.patch.set_facecolor('#1a1a2e')
    fig.suptitle('Confusion Matrices', color='white', fontsize=14, fontweight='bold')
    axes_flat = np.array(axes).flatten() if len(cm_files) > 1 else [axes]

    for ax, fp in zip(axes_flat, cm_files):
        img = mpimg.imread(str(fp))
        ax.imshow(img)
        ax.axis('off')

    for ax in axes_flat[len(cm_files):]:
        ax.set_visible(False)

    plt.tight_layout()
    plt.savefig('outputs/figures/colab_confusion_matrices.png', dpi=150,
                bbox_inches='tight', facecolor='#1a1a2e')
    plt.show()
    print(f'✅ Displayed {len(cm_files)} confusion matrices.')

## 🔍 Stage 8 — Explainability

In [ ]:
# ── Generate GNNExplainer + feature importance ─────────────────────────────
!python scripts/explain.py \
    --attack_type {ATTACK_TYPE} \
    --snapshots   {N_SNAPSHOTS}

In [ ]:
# ── Display explanation figures ────────────────────────────────────────────
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig_dir = Path('outputs/figures')
explain_figs = sorted([f for f in fig_dir.glob('*.png')
                       if any(k in f.name.lower()
                              for k in ['explain','shap','attn','importance','temporal'])])

if explain_figs:
    ncols = 2
    nrows = (len(explain_figs) + 1) // 2
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows * 5))
    fig.patch.set_facecolor('#1a1a2e')
    axes = np.array(axes).flatten() if nrows > 1 else [axes] * 2

    for ax, fp in zip(axes, explain_figs):
        img = mpimg.imread(str(fp))
        ax.imshow(img); ax.axis('off')
        ax.set_title(fp.stem, color='white', fontsize=9)
    for ax in axes[len(explain_figs):]:
        ax.set_visible(False)
    plt.tight_layout()
    plt.show()
else:
    print('No explanation figures found. Run explain.py first.')

## 🧪 Stage 9 — Ablation Study

In [ ]:
# ── Run ablation study ────────────────────────────────────────────────────
# study options: all | components | attacks | ratios | topology
ABLATION_STUDY = 'all'   # change to a specific study for speed

!python scripts/ablation.py \
    --study    {ABLATION_STUDY} \
    --epochs   {N_EPOCHS} \
    --snapshots {N_SNAPSHOTS}

In [ ]:
# ── Visualise ablation results ─────────────────────────────────────────────
import json, matplotlib.pyplot as plt, numpy as np
from pathlib import Path

abl_file = Path('outputs/tables/ablation_full_results.json')
if not abl_file.exists():
    print('Ablation results not found. Run ablation.py first.'); raise SystemExit

results = json.loads(abl_file.read_text())

studies = {k: v for k, v in results.items() if isinstance(v, dict)}
fig, axes = plt.subplots(1, len(studies), figsize=(5 * len(studies), 5))
if len(studies) == 1:
    axes = [axes]
fig.patch.set_facecolor('#1a1a2e')
fig.suptitle('Ablation Study — F1 Scores', color='white', fontsize=14, fontweight='bold')

PALETTE2 = ['#6C63FF','#00D4FF','#FFB300','#FF7043','#00E676','#F06292']

for ax, (study_name, study_data) in zip(axes, studies.items()):
    ax.set_facecolor('#16213e')
    ax.tick_params(colors='white')
    ax.xaxis.label.set_color('white'); ax.yaxis.label.set_color('white')
    ax.title.set_color('white')
    for sp in ax.spines.values(): sp.set_edgecolor('#2D3250')

    configs = list(study_data.keys())
    f1_vals = [study_data[c].get('f1', 0) if isinstance(study_data[c], dict)
               else study_data[c] for c in configs]
    bars = ax.bar(range(len(configs)), f1_vals,
                  color=PALETTE2[:len(configs)], alpha=0.85,
                  edgecolor='white', linewidth=0.3)
    ax.set_xticks(range(len(configs)))
    ax.set_xticklabels([c.replace('_',' ') for c in configs],
                        rotation=30, ha='right', color='white', fontsize=8)
    ax.set_ylim(0, 1.1)
    ax.set_title(study_name.replace('_',' ').title(), fontweight='bold')

    for bar, v in zip(bars, f1_vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.02, f'{v:.3f}',
                ha='center', va='bottom', color='white', fontsize=8)

plt.tight_layout()
plt.savefig('outputs/figures/colab_ablation.png', dpi=150,
            bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print('✅ Ablation chart saved.')

## 💾 Stage 10 — Package & Download Results

In [ ]:
# ── Zip all outputs ───────────────────────────────────────────────────────
import shutil, os
from pathlib import Path
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M')
archive_name = f'/content/byzantine_results_{timestamp}'

outputs_dir = Path('outputs')
if outputs_dir.exists():
    shutil.make_archive(archive_name, 'zip', 'outputs')
    archive_path = archive_name + '.zip'
    size_mb = os.path.getsize(archive_path) / 1e6
    print(f'✅ Archive created: {archive_path}  ({size_mb:.1f} MB)')
    print('\nContents:')
    for f in sorted(outputs_dir.rglob('*')):
        if f.is_file():
            print(f'  {f.relative_to(outputs_dir)}  ({f.stat().st_size/1024:.0f} KB)')
else:
    print('outputs/ directory not found.')

In [ ]:
# ── Download the zip ──────────────────────────────────────────────────────
from google.colab import files
import glob

archives = sorted(glob.glob('/content/byzantine_results_*.zip'))
if archives:
    latest = archives[-1]
    print(f'Downloading: {latest}')
    files.download(latest)
else:
    print('No archive found. Run the packaging cell above first.')

In [ ]:
# ── (Alternative) Save results to Google Drive ────────────────────────────
import shutil, glob

drive_output = '/content/drive/MyDrive/byzantine_results'

archives = sorted(glob.glob('/content/byzantine_results_*.zip'))
if archives:
    shutil.copy(archives[-1], drive_output + '.zip')
    print(f'✅ Saved to Google Drive: {drive_output}.zip')
else:
    print('No archive found. Run the packaging cell first.')

## ⚡ Bonus — Run Full Pipeline in One Command

If you want to run everything end-to-end without interruption:

In [ ]:
# ── Full pipeline (one cell) ───────────────────────────────────────────────
# Adjust N_SNAPSHOTS, N_EPOCHS, and N_EVAL_SEEDS at the top first.

import subprocess, sys

STAGES = [
    ('Preprocessing',       f'python scripts/preprocess.py'),
    ('Build Graphs',        f'python scripts/build_graphs.py --attack_type {ATTACK_TYPE} --graph_type {GRAPH_TYPE} --n_snapshots {N_SNAPSHOTS}'),
    ('Train Baselines',     f'python scripts/train_baselines.py --attack_type {ATTACK_TYPE} --epochs {N_EPOCHS} --snapshots {N_SNAPSHOTS}'),
    ('Train GAT+GRU',       f'python scripts/train_gnn_temporal.py --model gat --attack_type {ATTACK_TYPE} --graph_type {GRAPH_TYPE} --epochs {N_EPOCHS} --snapshots {N_SNAPSHOTS}'),
    ('Train GraphSAGE+GRU', f'python scripts/train_gnn_temporal.py --model graphsage --attack_type {ATTACK_TYPE} --graph_type {GRAPH_TYPE} --epochs {N_EPOCHS} --snapshots {N_SNAPSHOTS}'),
    ('Evaluate',            f'python scripts/evaluate.py --attack_type {ATTACK_TYPE} --graph_type {GRAPH_TYPE} --snapshots {N_SNAPSHOTS} --eval_seeds {N_EVAL_SEEDS}'),
    ('Explainability',      f'python scripts/explain.py --attack_type {ATTACK_TYPE} --snapshots {N_SNAPSHOTS}'),
    ('Ablation',            f'python scripts/ablation.py --study components --epochs {N_EPOCHS} --snapshots {N_SNAPSHOTS}'),
]

env = {**__import__('os').environ, 'PYTHONIOENCODING': 'utf-8'}

for stage_name, cmd in STAGES:
    print(f'\n{'='*60}')
    print(f'  STAGE: {stage_name}')
    print(f'{'='*60}')
    result = subprocess.run(cmd, shell=True, env=env, cwd='/content/iod')
    if result.returncode != 0:
        print(f'\n❌ Stage "{stage_name}" FAILED (exit code {result.returncode})')
        print('   Fix the error above and re-run from this stage.')
        break
    else:
        print(f'✅ {stage_name} complete.')

print('\n🎉 Pipeline complete! Run the download cell to get your results.')